In [16]:
import pandas as pd
from sklearn.datasets import fetch_openml

# Загрузка данных
data = fetch_openml(name="adult", version=2, as_frame=True)
X = data.data
y = (data.target == '>50K').astype(int)  # Бинарная целевая переменная

df = pd.concat([X, y.rename('income')], axis=1)

In [17]:
df['age_binned'] = pd.cut(df['age'], bins=5, labels=False)

In [27]:
from CHAID import Tree

# {'признак': 'тип'}
variable_types = {
    'age': 'nominal',
    'workclass': 'nominal',
    'education': 'nominal',
    'marital-status': 'nominal',
    'occupation': 'nominal',
    'relationship': 'nominal',
    'race': 'nominal',
    'sex': 'nominal',
    'native-country': 'nominal',
    'hours-per-week': 'nominal'
}

tree = Tree.from_pandas_df(
    df, 
    variable_types,
    d_variable='income',
    max_depth=5,
    min_parent_node_size=100
)

# Визуализация дерева
tree.print_tree()

([], {np.int64(0): np.float64(37155.0), np.int64(1): np.float64(11687.0)}, (relationship, p=0.0, score=10084.03745658185, groups=[['Husband', 'Wife'], ['Not-in-family'], ['Other-relative'], ['Own-child'], ['Unmarried']]), dof=4))
|-- (['Husband', 'Wife'], {np.int64(0): np.float64(12108.0), np.int64(1): np.float64(9939.0)}, (education, p=0.0, score=3679.497332707954, groups=[['10th', '11th'], ['12th'], ['1st-4th', '5th-6th', '7th-8th', '9th', 'Preschool'], ['Assoc-acdm'], ['Assoc-voc', 'Some-college'], ['Bachelors'], ['Doctorate', 'Prof-school'], ['HS-grad'], ['Masters']]), dof=8))
|   |-- (['10th', '11th'], {np.int64(0): np.float64(890.0), np.int64(1): np.float64(150.0)}, (age, p=8.636087578105987e-20, score=95.56627380825566, groups=[[18, 19, 20, 21, 30, 35, 62, 68, 69, 70, 72, 73, 74, 76, 77, 78, 79, 80, 81, 89, 90], [22, 52, 25, 31, 37, 47, 63, 32, 49, 53], [23, 26, 43, 45, 27, 39, 33, 36, 24, 34, 54, 42, 67], [28, 29, 57, 56, 41, 48, 55, 40, 60, 66, 71, 58, 50, 59], [38, 44, 51, 46

In [28]:
from sklearn.model_selection import train_test_split

# Разделение данных
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

# Построение дерева на train
train_tree = Tree.from_pandas_df(train_df, variable_types, 'income', max_depth=5)

# Предсказание на test
def predict(tree, row):
    node = tree.tree_store[0]
    while node.split is not None:
        col = node.split.column
        val = row[col]
        for i, group in enumerate(node.split.groups):
            if val in group:
                node = tree.tree_store[node.children[i]]
                break
    return max(node.members.items(), key=lambda x: x[1])[0]

test_df['predicted'] = test_df.apply(lambda x: predict(train_tree, x), axis=1)
accuracy = (test_df['predicted'] == test_df['income']).mean()
print(f"Accuracy: {accuracy:.2f}")

AttributeError: 'Split' object has no attribute 'groups'